<a href="https://colab.research.google.com/github/ravimahendrakar/jax-sample-colabs/blob/main/JAX_Ecosystem_for_Image_Classification_A_Deep_Dive_for_Developers_%26_Researchers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Copyright 2025 Google LLC.

Licensed under the Apache License, Version 2.0 (the "License");

# JAX Ecosystem for Image Classification: A Deep Dive for Developers & Researchers

Introduction: Harnessing the Power of JAX for Machine Learning Research

Welcome to this demonstration of the JAX ecosystem for building and training a deep learning model for image classification. This notebook is designed for **developers and researchers** who are interested in leveraging JAX's unparalleled performance, its elegant functional programming paradigm, and its powerful ecosystem for cutting-edge machine learning.

Why JAX and its Ecosystem?

**JAX** offers a unique combination of automatic differentiation, JIT compilation (via XLA), and vectorization/parallelization capabilities, making it exceptionally well-suited for high-performance numerical computing and machine learning research.

This notebook showcases the synergy of key libraries within the JAX ecosystem:

**Flax NNX:** A modern, declarative, and stateful neural network API built on JAX. It simplifies model definition, management of parameters, and integration with other JAX tools.

 *   **Optax:** A gradient processing and optimization library for JAX, providing a flexible and composable API for building sophisticated training loops with learning rate schedules, gradient clipping, and more.
 *   **Orbax:** A library for robust checkpointing, serialization, and sharding of JAX PyTrees. It's essential for saving and restoring model states, ensuring reproducibility and enabling fault-tolerant training.
 *   **TensorFlow Datasets (`tfds`):** Utilized for efficient data loading and preprocessing, seamlessly integrating with JAX by converting TensorFlow datasets to JAX arrays.

 Project Goal: Image Classification

We will train a Convolutional Neural Network (CNN) to classify images from the **CIFAR-100 dataset**, focusing on the 20 "superclasses." This end-to-end demonstration covers:

1.  **Environment Setup:** Installing and verifying JAX and its ecosystem.
2.  **Data Handling:** Loading and preprocessing data with `tf.data` and applying augmentation.
3.  **Model Building:** Defining a CNN using Flax NNX.
4.  **Training & Evaluation:** Implementing training loops with Optax and JAX's JIT.
5.  **Checkpointing:** Saving and loading model states with Orbax.
6.  **Inference:** Performing image classification on sample images using the trained model.



# Section 1: Let's begin by setting up our environment.

In [ ]:
# @title 1.1 Install Dependencies
# @markdown This cell installs all the necessary Python packages for the notebook.

# Ensure you have the latest versions for optimal performance and compatibility.
# The `--upgrade` flag ensures we get the most recent stable releases.
print("Installing JAX, Flax, Optax, Orbax, TensorFlow, and TensorFlow Datasets...")
!pip install --upgrade jax[cpu] flax optax orbax-checkpoint tensorflow tensorflow_datasets matplotlib tqdm Pillow -q

print("\nInstallation complete. Proceeding with imports.")

In [ ]:
# @title 1.2 Import Libraries
# @markdown Import all the essential libraries required for this demonstration.

import os
import sys
import subprocess # For checking Python version more robustly

# --- Core JAX Ecosystem Components ---
import jax
import jax.numpy as jnp
from flax import nnx
import optax
import orbax.checkpoint as ocp

# --- Data Handling and Visualization ---
import tensorflow as tf
import tensorflow_datasets as tfds
import numpy as np
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm # For progress bars

# --- Image Processing Utilities ---
from PIL import Image
import requests
from io import BytesIO

# --- Configuration ---
# Add the virtual environment's site-packages to sys.path if necessary
# (Though for Colab installs, this is usually handled by pip)
py_version = f"python{sys.version_info.major}.{sys.version_info.minor}"
# VENV_PATH = os.path.join(VENV_DIR, 'lib', py_version, 'site-packages')
# VENV_PATH = os.path.abspath(VENV_PATH)
# if VENV_PATH not in sys.path:
#     sys.path.insert(0, VENV_PATH)
#     print(f"✅ Added '{VENV_PATH}' to sys.path.")

print("✅ All necessary libraries imported successfully.")

In [ ]:
# @title 1.3 Verify JAX and Device Availability
# @markdown This cell checks JAX and Flax versions and confirms available hardware accelerators.

# --- JAX/Flax Version Verification ---
import flax # Added import here to ensure availability
import optax # Added import here to ensure availability
import orbax.checkpoint as ocp # Added import here to ensure availability
import tensorflow as tf # Added import here to ensure availability
import jax # Added import here to ensure availability

print(f"JAX version: {jax.__version__}")
print(f"Flax version: {flax.__version__}")
print(f"Optax version: {optax.__version__}")
print(f"Orbax Checkpoint version: {ocp.__version__}")
print(f"TensorFlow version: {tf.__version__}")

# --- Device Availability Check ---
print("\n--- JAX Device Information ---")
try:
    devices = jax.devices()
    print(f"Available JAX devices: {len(devices)}")
    for i, d in enumerate(devices):
        print(f"  Device {i}: Platform='{d.platform}', Kind='{d.device_kind}'")

    # Informative message about TPU availability
    if any(d.platform == 'tpu' for d in devices):
        print("\n🚀 Success! TPU detected and ready to use.")
    else:
        print("\n⚠️ Warning: No TPU detected. The code will run on CPU/GPU.")
        print("   JAX ecosystem libraries are designed to work seamlessly across accelerators.")
        print("   For this demonstration, CPU/GPU execution is sufficient and expected.")

except Exception as e:
    print(f"❌ An error occurred while checking JAX devices: {e}")
    print("   JAX might not be installed correctly or there's an issue with the environment.")

# --- TensorFlow GPU Memory Configuration ---
# It's good practice to prevent TensorFlow from allocating GPU memory
# if JAX is intended to use it, or if running on systems where TensorFlow
# might manage devices differently.
tf.config.experimental.set_visible_devices([], 'GPU')
tf.config.experimental.set_visible_devices([], 'TPU') # Ensure TF doesn't claim TPUs if JAX needs them.

print("\n✅ JAX and device information verified. TensorFlow GPU/TPU memory usage configured.")

In [ ]:
# @title 1.4 Global Configurations
# @markdown Set up global parameters for training and data handling.

# --- Training Parameters ---
EPOCHS = 15  # Number of full passes over the training data
BATCH_SIZE = 128  # Number of samples per training/inference batch

# --- Paths ---
# Define directories for storing checkpoints and the final model
# Checkpoints are saved incrementally during training for fault tolerance.
# The final model is saved separately for clean inference.
checkpoint_base_dir = os.path.join(os.getcwd(), "checkpoints")
final_model_save_dir = os.path.join(os.getcwd(), "final_model")

# Ensure directories exist (Orbax CheckpointManager will create them if not)
os.makedirs(checkpoint_base_dir, exist_ok=True)
os.makedirs(final_model_save_dir, exist_ok=True) # Created for clarity, Orbax will write here

print(f"Configured Epochs: {EPOCHS}")
print(f"Configured Batch Size: {BATCH_SIZE}")
print(f"Checkpoint Directory: {checkpoint_base_dir}")
print(f"Final Model Save Directory: {final_model_save_dir}")

print("\n✅ Global configurations set.")

In [ ]:
# @title 2.1 Understanding tf.data and Preprocessing
# @markdown Learn how to load and preprocess datasets using TensorFlow Datasets and integrate them with JAX.

# --- Data Augmentation Layer ---
# We define a small Keras Sequential model that will only perform data augmentation.
# This allows us to leverage TensorFlow's optimized data pipeline for these operations.
data_augmentation_layer = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1),
], name="data_augmentation")

# --- CIFAR-100 Superclass Names ---
# CIFAR-100 has 100 classes, grouped into 20 superclasses.
# We'll use these superclasses for our classification task.
CIFAR100_SUPERCLASS_NAMES = [
    'aquatic mammals', 'fish', 'flowers', 'food containers', 'fruit and vegetables',
    'household electrical devices', 'household furniture', 'insects', 'large carnivores',
    'large man-made outdoor things', 'large natural outdoor scenes', 'large omnivores and herbivores',
    'medium-sized mammals', 'non-inline_images', 'ocean', 'people', 'reptiles', 'small mammals',
    'trees', 'vehicles 1', 'vehicles 2'
]
NUM_CLASSES = len(CIFAR100_SUPERCLASS_NAMES) # Total number of superclasses

print(f"Using {NUM_CLASSES} superclasses for classification.")

def load_and_preprocess_cifar100(batch_size: int, augment: bool = True):
    """
    Loads and preprocesses the CIFAR-100 dataset for 20-superclass classification.

    Args:
        batch_size: The desired batch size for the dataset.
        augment: Whether to apply data augmentation to the training set.

    Returns:
        A tuple containing the preprocessed training dataset and test dataset.
    """
    # --- Preprocessing Function ---
    def preprocess_element(element):
        """Normalizes image pixels and selects the coarse label."""
        image = tf.cast(element['image'], tf.float32) / 255.0  # Normalize to [0, 1]
        # 'coarse_label' corresponds to the 20 superclasses in CIFAR-100.
        label = element['coarse_label']
        return {'image': image, 'label': label}

    # --- Load Training Dataset ---
    print("Loading CIFAR-100 training dataset...")
    train_ds = tfds.load('cifar100', split='train', as_supervised=False)
    train_ds = train_ds.map(preprocess_element, num_parallel_calls=tf.data.AUTOTUNE)

    # --- Data Augmentation for Training ---
    if augment:
        print("Applying data augmentation to the training set...")
        train_ds = train_ds.map(
            lambda x: {**x, 'image': data_augmentation_layer(x['image'], training=True)},
            num_parallel_calls=tf.data.AUTOTUNE
        )

    # --- Batching and Prefetching for Training ---
    # Shuffling is crucial for good training performance.
    # Prefetching helps overlap data preprocessing and model execution.
    train_ds = train_ds.shuffle(10000).batch(batch_size).prefetch(tf.data.AUTOTUNE)

    # --- Load Test Dataset ---
    print("Loading CIFAR-100 test dataset...")
    test_ds = tfds.load('cifar100', split='test', as_supervised=False)
    test_ds = test_ds.map(preprocess_element, num_parallel_calls=tf.data.AUTOTUNE)

    # --- Batching and Prefetching for Testing ---
    # No shuffling or augmentation for the test set.
    test_ds = test_ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

    print("Dataset loading and preprocessing complete.")
    return train_ds, test_ds

# Load the datasets using the configured batch size.
train_dataset, test_dataset = load_and_preprocess_cifar100(BATCH_SIZE)

# --- Verify Dataset Structure ---
print("\n--- Dataset Verification ---")
# Inspect the structure and shapes of a sample batch
sample_batch = next(iter(train_dataset))
print(f"Sample batch structure: {train_dataset.element_spec}")
print(f"Sample batch image shape: {sample_batch['image'].shape}")
print(f"Sample batch label shape: {sample_batch['label'].shape}")

print("\n✅ Dataset loading and verification complete.")

# Section 2: Data Loading and Preprocessing

This section focuses on efficiently loading and preparing the CIFAR-100 dataset. We'll use TensorFlow Datasets (tfds) for its convenience and broad dataset availability, then adapt the data for JAX/Flax. We'll also implement data augmentation to improve model generalization.

In [ ]:
# @title 2.1 Understanding tf.data and Preprocessing
# @markdown Learn how to load and preprocess datasets using TensorFlow Datasets and integrate them with JAX.

# --- Data Augmentation Layer ---
# We define a small Keras Sequential model that will only perform data augmentation.
# This allows us to leverage TensorFlow's optimized data pipeline for these operations.
data_augmentation_layer = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1),
], name="data_augmentation")

# --- CIFAR-100 Superclass Names ---
# CIFAR-100 has 100 classes, grouped into 20 superclasses.
# We'll use these superclasses for our classification task.
CIFAR100_SUPERCLASS_NAMES = [
    'aquatic mammals', 'fish', 'flowers', 'food containers', 'fruit and vegetables',
    'household electrical devices', 'household furniture', 'insects', 'large carnivores',
    'large man-made outdoor things', 'large natural outdoor scenes', 'large omnivores and herbivores',
    'medium-sized mammals', 'non-inline_images', 'ocean', 'people', 'reptiles', 'small mammals',
    'trees', 'vehicles 1', 'vehicles 2'
]
NUM_CLASSES = len(CIFAR100_SUPERCLASS_NAMES) # Total number of superclasses

print(f"Using {NUM_CLASSES} superclasses for classification.")

def load_and_preprocess_cifar100(batch_size: int, augment: bool = True):
    """
    Loads and preprocesses the CIFAR-100 dataset for 20-superclass classification.

    Args:
        batch_size: The desired batch size for the dataset.
        augment: Whether to apply data augmentation to the training set.

    Returns:
        A tuple containing the preprocessed training dataset and test dataset.
    """
    # --- Preprocessing Function ---
    def preprocess_element(element):
        """Normalizes image pixels and selects the coarse label."""
        image = tf.cast(element['image'], tf.float32) / 255.0  # Normalize to [0, 1]
        # 'coarse_label' corresponds to the 20 superclasses in CIFAR-100.
        label = element['coarse_label']
        return {'image': image, 'label': label}

    # --- Load Training Dataset ---
    print("Loading CIFAR-100 training dataset...")
    train_ds = tfds.load('cifar100', split='train', as_supervised=False)
    train_ds = train_ds.map(preprocess_element, num_parallel_calls=tf.data.AUTOTUNE)

    # --- Data Augmentation for Training ---
    if augment:
        print("Applying data augmentation to the training set...")
        train_ds = train_ds.map(
            lambda x: {**x, 'image': data_augmentation_layer(x['image'], training=True)},
            num_parallel_calls=tf.data.AUTOTUNE
        )

    # --- Batching and Prefetching for Training ---
    # Shuffling is crucial for good training performance.
    # Prefetching helps overlap data preprocessing and model execution.
    train_ds = train_ds.shuffle(10000).batch(batch_size).prefetch(tf.data.AUTOTUNE)

    # --- Load Test Dataset ---
    print("Loading CIFAR-100 test dataset...")
    test_ds = tfds.load('cifar100', split='test', as_supervised=False)
    test_ds = test_ds.map(preprocess_element, num_parallel_calls=tf.data.AUTOTUNE)

    # --- Batching and Prefetching for Testing ---
    # No shuffling or augmentation for the test set.
    test_ds = test_ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

    print("Dataset loading and preprocessing complete.")
    return train_ds, test_ds

# Load the datasets using the configured batch size.
train_dataset, test_dataset = load_and_preprocess_cifar100(BATCH_SIZE)

# --- Verify Dataset Structure ---
print("\n--- Dataset Verification ---")
# Inspect the structure and shapes of a sample batch
sample_batch = next(iter(train_dataset))
print(f"Sample batch structure: {train_dataset.element_spec}")
print(f"Sample batch image shape: {sample_batch['image'].shape}")
print(f"Sample batch label shape: {sample_batch['label'].shape}")

print("\n✅ Dataset loading and verification complete.")

In [ ]:
# @title 2.2 Visualize Sample Data
# @markdown Display a few images from the dataset to get a visual understanding of the data.

# Retrieve a sample batch to visualize.
try:
    sample_batch = next(iter(train_dataset))
    images = sample_batch['image'].numpy()
    labels = sample_batch['label'].numpy()

    # Determine the number of images to display.
    num_images_to_show = min(5, images.shape[0])

    # Create a figure and axes for plotting.
    fig, axes = plt.subplots(1, num_images_to_show, figsize=(12, 3))
    if num_images_to_show == 1: # Handle case where only one image is shown
        axes = [axes]

    # Plot each image with its corresponding superclass label.
    for i in range(num_images_to_show):
        ax = axes[i]
        img = images[i]
        label_index = labels[i]
        ax.imshow(img)
        ax.set_title(f"Label: {CIFAR100_SUPERCLASS_NAMES[label_index]}", fontsize=9)
        ax.axis('off') # Hide axes ticks and labels

    plt.tight_layout()
    plt.suptitle("Sample CIFAR-100 Superclass Images", y=1.02)
    plt.show()

except StopIteration:
    print("Error: Could not retrieve a sample batch from the dataset.")
except Exception as e:
    print(f"An error occurred during visualization: {e}")

print("\n✅ Sample data visualization complete.")

# Section 3: Model Definition with Flax NNX
This section introduces Flax NNX, a modern neural network library within the JAX ecosystem. We'll define a CNN model and set up an optimizer using Optax, demonstrating how these tools facilitate building complex ML models efficiently.

In [ ]:
# @title 3.1 Introduction to Flax NNX
# @markdown Learn why Flax NNX is a powerful choice for building and managing neural network state in JAX.

# ### Flax NNX: Declarative and Stateful Modules

# **Flax NNX** is designed for building neural networks in a more declarative and stateful manner compared to older functional approaches. Key features beneficial for developers and researchers include:
# *   **Object-Oriented Structure:** Models are defined as Python classes inheriting from `nnx.Module`, making them intuitive to work with.
# *   **State Management:** NNX modules explicitly manage their state (parameters, variables, RNG streams) through attributes, simplifying state tracking and manipulation.
# *   **Composability:** Modules can be easily nested and composed, allowing for the construction of complex architectures from simpler building blocks.
# *   **Built-in RNG Handling:** NNX modules automatically manage JAX random number generator (RNG) streams, crucial for parameter initialization and stochastic layers like Dropout.
# *   **Integration:** Seamlessly integrates with JAX transformations like `jit`, `grad`, and `vmap`, as well as optimization libraries like Optax and checkpointing tools like Orbax.

print("Flax NNX provides a modern, stateful approach to building neural networks in JAX.")

In [ ]:
# @title 3.2 Define CNN Architecture using Flax NNX
# @markdown Define a Convolutional Neural Network (CNN) using Flax NNX modules.

# We define a standard CNN architecture suitable for image classification tasks.
# NNX's `Rngs` object is used to manage random number streams for initialization.
class CNN(nnx.Module):
    """A simple Convolutional Neural Network using Flax NNX."""

    def __init__(self, num_classes: int, *, rngs: nnx.Rngs):
        """Initializes the CNN model.

        Args:
            num_classes: The number of output classes for the classifier.
            rngs: An `nnx.Rngs` object to manage JAX random number streams.
                  'params' is used for parameter initialization.
                  'dropout' is used for the dropout layer.
        """
        # Convolutional Layers
        self.conv1 = nnx.Conv(in_features=3, out_features=32, kernel_size=(3, 3), rngs=rngs)
        # BatchNorm handles state (mean, variance) and uses `use_running_average` for inference vs training.
        self.bn1 = nnx.BatchNorm(num_features=32, use_running_average=True, rngs=rngs)

        self.conv2 = nnx.Conv(in_features=32, out_features=64, kernel_size=(3, 3), rngs=rngs)
        self.bn2 = nnx.BatchNorm(num_features=64, use_running_average=True, rngs=rngs)

        self.conv3 = nnx.Conv(in_features=64, out_features=128, kernel_size=(3, 3), rngs=rngs)
        self.bn3 = nnx.BatchNorm(num_features=128, use_running_average=True, rngs=rngs)

        # Fully Connected Layers
        # The input dimension to linear1 is determined by the output of the last conv layer,
        # after pooling: (32, 32) -> (16, 16) -> (8, 8) -> (4, 4). Output channels: 128.
        # So, input features = 128 * 4 * 4 = 2048.
        self.linear1 = nnx.Linear(in_features=128 * 4 * 4, out_features=512, rngs=rngs)
        # Dropout layer for regularization. `deterministic=True` during inference/evaluation.
        self.dropout = nnx.Dropout(rate=0.5, rngs=rngs)

        self.linear2 = nnx.Linear(in_features=512, out_features=num_classes, rngs=rngs)

    def __call__(self, x: jax.Array, training: bool):
        """Defines the forward pass of the CNN.

        Args:
            x: The input tensor (images).
            training: A boolean indicating whether the model is in training mode.
                      This affects BatchNorm and Dropout behavior.

        Returns:
            The output logits from the classifier.
        """
        # Apply Conv -> BatchNorm -> ReLU -> MaxPool
        x = self.conv1(x)
        # BatchNorm behavior is controlled by 'training' flag (use_running_average).
        x = self.bn1(x, use_running_average=not training)
        x = nnx.relu(x)
        x = nnx.max_pool(x, window_shape=(2, 2), strides=(2, 2))

        x = self.conv2(x)
        x = self.bn2(x, use_running_average=not training)
        x = nnx.relu(x)
        x = nnx.max_pool(x, window_shape=(2, 2), strides=(2, 2))

        x = self.conv3(x)
        x = self.bn3(x, use_running_average=not training)
        x = nnx.relu(x)
        x = nnx.max_pool(x, window_shape=(2, 2), strides=(2, 2))

        # Flatten the output for the fully connected layers.
        # x.shape[0] is the batch size, -1 infers the remaining dimension.
        x = x.reshape(x.shape[0], -1)

        x = self.linear1(x)
        x = nnx.relu(x)
        # Dropout is only applied during training.
        x = self.dropout(x, deterministic=not training)

        x = self.linear2(x)
        return x

print("CNN model architecture defined.")

In [ ]:
# @title 3.3 Model Instantiation and Optimizer Setup
# @markdown Create an instance of the CNN model and configure the optimizer using Optax.

# --- Initialize Model Components ---
# We need a JAX PRNGKey for parameter initialization and for the dropout stream.
key = jax.random.PRNGKey(0)
# Create an Rngs object, specifying the streams we'll use.
# 'params' for weight/bias initialization, 'dropout' for dropout sampling.
rngs = nnx.Rngs(params=key, dropout=jax.random.split(key)[1])

# Instantiate the CNN model.
# Pass the number of classes and the initialized RNG streams.
model = CNN(num_classes=NUM_CLASSES, rngs=rngs)
print("CNN model instantiated.")

# --- Learning Rate Schedule ---
# A smooth learning rate schedule often improves training stability and performance.
# We use a warmup phase followed by a cosine decay.
num_train_steps = len(train_dataset) * EPOCHS
# Calculate steps for warmup (e.x., 10% of total steps)
warmup_steps = int(num_train_steps * 0.1)

lr_schedule = optax.warmup_cosine_decay_schedule(
    init_value=0.0,           # Start learning rate
    peak_value=1e-3,          # Maximum learning rate
    warmup_steps=warmup_steps,
    decay_steps=num_train_steps,
    end_value=1e-5            # Minimum learning rate
)
print(f"Learning rate schedule configured: {warmup_steps} warmup steps.")

# --- Optimizer Setup ---
# We use the Adam optimizer with the defined learning rate schedule.
# NNX's `ModelAndOptimizer` class conveniently bundles the model and its optimizer.
# This is beneficial for state management and checkpointing.
optimizer = nnx.ModelAndOptimizer(model, optax.adam(learning_rate=lr_schedule))
print("Optax Adam optimizer with learning rate schedule configured.")

# Note: For single-device execution, explicit sharding (`jax.sharding`) is not required.
# NNX handles the state management internally for the current device context.

print("\n✅ Model instantiation and optimizer setup complete.")

# Section 4: Training and Evaluation Loops
This section details the core logic for training and evaluating the neural network. We'll define the loss function, accuracy metric, and the step functions, leveraging JAX's jit for performance optimization.

In [ ]:
# @title 4.1 Loss Function and Accuracy Metric
# @markdown Define the functions for calculating loss and accuracy.

# --- Loss Function ---
def cross_entropy_loss(logits: jax.Array, labels: jax.Array):
    """Calculates the cross-entropy loss for classification.

    Args:
        logits: The raw output scores from the model (before softmax).
        labels: The ground truth class indices.

    Returns:
        The cross-entropy loss for each sample in the batch.
    """
    # Convert labels to one-hot encoding for compatibility with cross-entropy.
    one_hot_labels = jax.nn.one_hot(labels, num_classes=NUM_CLASSES)
    # Calculate cross-entropy: -sum(y_true * log(softmax(y_pred)))
    return -jnp.sum(one_hot_labels * jax.nn.log_softmax(logits), axis=-1)

# --- Accuracy Metric ---
def accuracy(logits: jax.Array, labels: jax.Array):
    """Calculates the classification accuracy.

    Args:
        logits: The raw output scores from the model.
        labels: The ground truth class indices.

    Returns:
        The mean accuracy over the batch.
    """
    # Determine the predicted class by finding the index of the maximum logit.
    predictions = jnp.argmax(logits, axis=-1)
    # Compute the mean of correctly predicted samples.
    return jnp.mean(predictions == labels)

print("Loss and accuracy functions defined.")

In [ ]:
# @title 4.2 Training Step (`train_step`) with `nnx.jit`
# @markdown Implement the `train_step` function, optimized with JAX's Just-In-Time compilation.

# @markdown **Explanation of `nnx.jit`:**
# @markdown JAX's `jit` (Just-In-Time compilation) compiles Python/NumPy code into highly optimized XLA (Accelerated Linear Algebra) computations. For NNX, `nnx.jit` compiles methods or functions that operate on NNX modules, automatically handling state updates and ensuring efficient execution on the target device (CPU/GPU).

@nnx.jit
def train_step(model: CNN, optimizer: nnx.Optimizer, batch: dict):
    """Performs a single training step.

    Args:
        model: The Flax NNX model instance.
        optimizer: The Optax optimizer wrapped by NNX's ModelAndOptimizer.
        batch: A dictionary containing 'image' and 'label' for the current batch.

    Returns:
        A tuple containing:
            - The updated model.
            - The updated optimizer.
            - A dictionary of metrics (loss, accuracy) for the step.
    """
    # --- Define Loss Function Closure ---
    # This function computes the loss given the model's current state.
    # `nnx.value_and_grad` will differentiate this function.
    def loss_fn(current_model):
        # Run the model in training mode (enables BatchNorm updates and Dropout).
        logits = current_model(batch['image'], training=True)
        # Calculate the mean loss over the batch.
        loss = jnp.mean(cross_entropy_loss(logits, batch['label']))
        return loss

    # --- Compute Loss and Gradients ---
    # `nnx.value_and_grad` returns the loss value and the gradients of the loss
    # with respect to the model's parameters.
    loss, grads = nnx.value_and_grad(loss_fn)(model)

    # --- Update Model and Optimizer ---
    # The `optimizer.update` method applies the computed gradients to update
    # the model's parameters and the optimizer's internal state (e.g., Adam's moments).
    optimizer.update(grads)

    # --- Calculate Metrics ---
    # Re-run the model (in inference mode for consistency) to get logits for accuracy calculation.
    # Note: This might involve recompilation if the input shapes or `training` flag change behavior significantly.
    # For efficiency, metrics can sometimes be calculated during the gradient computation itself if possible.
    metrics = {
        'loss': loss,
        # Calculate accuracy using the non-training pass of the model.
        'accuracy': accuracy(model(batch['image'], training=False), batch['label'])
    }

    # Return the updated model, optimizer, and metrics.
    return model, optimizer, metrics

print("`train_step` function defined and JIT-compiled.")

In [ ]:
# @title 4.3 Evaluation Step (`eval_step`) with `nnx.jit`
# @markdown Implement the `eval_step` function for model evaluation, also optimized with JAX's JIT.

@nnx.jit
def eval_step(model: CNN, batch: dict):
    """Performs a single evaluation step.

    Args:
        model: The Flax NNX model instance.
        batch: A dictionary containing 'image' and 'label' for the current batch.

    Returns:
        A dictionary of metrics (loss, accuracy) for the step.
    """
    # Run the model in evaluation mode (disables Dropout, uses running averages for BatchNorm).
    logits = model(batch['image'], training=False)
    # Calculate the mean loss over the batch.
    loss = jnp.mean(cross_entropy_loss(logits, batch['label']))
    # Calculate the accuracy.
    acc = accuracy(logits, batch['label'])

    return {'loss': loss, 'accuracy': acc}

print("`eval_step` function defined and JIT-compiled.")

In [ ]:
# @title 4.4 Main Training Loop
# @markdown Orchestrates the training process over multiple epochs, including checkpointing.

print(f"\nStarting training for {EPOCHS} epochs...")

# --- Checkpoint Manager Setup ---
# Orbax CheckpointManager is used for managing multiple checkpoints.
# `max_to_keep=5` means only the latest 5 checkpoints will be retained.
# `create=True` ensures the directory is created if it doesn't exist.
# `ocp.test_utils.erase_and_create_empty` is used here for a clean run,
# ensuring we start with a fresh checkpoint directory each time the cell runs.
# In a real research scenario, you'd typically omit `erase_and_create_empty`
# to allow resuming from existing checkpoints.
options = ocp.CheckpointManagerOptions(max_to_keep=5, create=True)
mngr = ocp.CheckpointManager(ocp.test_utils.erase_and_create_empty(checkpoint_base_dir), options=options)

# --- Training Loop ---
for epoch in range(EPOCHS):
    # Track metrics per epoch
    train_losses, train_accuracies = [], []

    # --- Training Phase ---
    # Iterate through the training dataset. tqdm provides a progress bar.
    for batch in tqdm(train_dataset, desc=f"Epoch {epoch + 1}/{EPOCHS} Training", leave=False):
        # Convert TensorFlow tensors to JAX arrays for use with JAX functions.
        jax_batch = jax.tree_util.tree_map(jnp.asarray, batch)

        # Execute the training step. NNX automatically manages state updates via `optimizer.update`.
        model, optimizer, metrics = train_step(model, optimizer, jax_batch)

        # Accumulate metrics for the epoch.
        train_losses.append(metrics['loss'])
        train_accuracies.append(metrics['accuracy'])

    # Calculate average training metrics for the epoch.
    avg_train_loss = np.mean(train_losses)
    avg_train_accuracy = np.mean(train_accuracies)

    # --- Evaluation Phase ---
    test_losses, test_accuracies = [], []
    # Iterate through the test dataset for evaluation.
    for batch in tqdm(test_dataset, desc=f"Epoch {epoch + 1}/{EPOCHS} Evaluating", leave=False):
        jax_batch = jax.tree_util.tree_map(jnp.asarray, batch)
        metrics = eval_step(model, jax_batch)
        test_losses.append(metrics['loss'])
        test_accuracies.append(metrics['accuracy'])

    # Calculate average test metrics for the epoch.
    avg_test_loss = np.mean(test_losses)
    avg_test_accuracy = np.mean(test_accuracies)

    # --- Learning Rate Logging ---
    # Retrieve the current step count from the optimizer state to determine the current LR.
    # The optimizer state is often a tuple; `count` is typically in the first element for Adam.
    step_count_array = optimizer.opt_state[0].count
    current_lr = lr_schedule(int(step_count_array)) # Use step_count_array to get LR from schedule

    # --- Print Epoch Summary ---
    print(f"Epoch {epoch + 1}/{EPOCHS}: "
          f"Train Loss={avg_train_loss:.4f}, Train Acc={avg_train_accuracy:.4f} | "
          f"Test Loss={avg_test_loss:.4f}, Test Acc={avg_test_accuracy:.4f} | LR={current_lr:.6f}")

    # --- Checkpointing ---
    # Prepare the state to be saved. NNX's `nnx.state()` extracts the state from modules.
    # We save both the model and optimizer states.
    checkpoint_data = {'model': nnx.state(model), 'optimizer': nnx.state(optimizer)}
    # Save the checkpoint asynchronously. `mngr.save` takes the step number and checkpoint item.
    mngr.save(epoch + 1, args=ocp.args.StandardSave(checkpoint_data))
    # Wait for the save operation to complete.
    mngr.wait_until_finished()
    print(f"Checkpoint saved for epoch {epoch + 1} to {checkpoint_base_dir}.")

# --- Cleanup and Final Model Saving ---
mngr.close() # Close the checkpoint manager.
print("\n✅ Training loop finished.")

# --- Save Final Model for Inference ---
# This section saves a clean version of the model, without optimizer state,
# which is ideal for deployment or subsequent inference tasks.
print(f"\n... Saving final, clean model for inference to '{final_model_save_dir}'.")

# Instantiate a PyTreeCheckpointer, which is simpler for saving single PyTrees.
final_checkpointer = ocp.PyTreeCheckpointer()

# Extract only the model's state. Optimizer state is not needed for inference.
final_model_state = nnx.state(model)

# Save the model state PyTree directly to the specified directory.
# Use force=True to overwrite if the directory already exists.
final_checkpointer.save(final_model_save_dir, final_model_state, force=True)

print(f"✅ Final inference model saved successfully to '{final_model_save_dir}'.")

print("\nTraining and model saving process complete.")

# Section 5: Checkpointing with Orbax
Reproducibility and fault tolerance are paramount in research. This section demonstrates how Orbax provides a robust solution for saving and loading model states, ensuring progress is not lost and experiments can be reliably reproduced.

In [ ]:
# @title 5.1 Introduction to Orbax Checkpointing
# @markdown Understand Orbax's role in reliably managing model and optimizer states.

# ### Orbax: Robust Checkpointing for JAX

# **Orbax Checkpointing** is a powerful library within the JAX ecosystem designed to handle the saving and restoring of complex JAX PyTrees. Its key advantages for researchers include:

# *   **Flexibility:** Supports various checkpointing formats and strategies, including saving entire PyTrees, specific subtrees, or using custom transformations.
# *   **Performance:** Optimized for efficiency, handling large models and potentially distributed setups.
# *   **Reliability:** Built to handle long-running training jobs, offering features like asynchronous saving and fault tolerance.
# *   **Integration with NNX:** Works seamlessly with `nnx.state()` to capture the complete state of NNX modules and `nnx.ModelAndOptimizer` for saving both model and optimizer states.

# In the previous training section, we used `ocp.CheckpointManager` to save checkpoints after each epoch.
# We also saved a final, clean model state for inference.
print("Orbax provides essential tools for managing model state in JAX for research reproducibility.")

# Section 6: Model Evaluation
This section provides a final evaluation of the trained model on the entire test dataset, giving a clear picture of its performance after training.

In [ ]:
# @title 6.1 Final Evaluation on Test Set
# @markdown Assess the model's performance on unseen data.

print("\n--- Final Model Evaluation ---")
print("Evaluating the trained model on the complete test dataset...")

# Initialize lists to store evaluation metrics.
final_test_losses, final_test_accuracies = [], []

# Iterate through the test dataset.
for batch in tqdm(test_dataset, desc="Final Evaluation", leave=False):
    # Convert TF tensors to JAX arrays.
    jax_batch = jax.tree_util.tree_map(jnp.asarray, batch)

    # Run the evaluation step.
    metrics = eval_step(model, jax_batch)

    # Accumulate metrics.
    final_test_losses.append(metrics['loss'])
    final_test_accuracies.append(metrics['accuracy'])

# Calculate and print the final average metrics.
# We use jnp.array for efficient computation if available, otherwise numpy.
final_test_loss = jnp.mean(jnp.array(final_test_losses))
final_test_accuracy = jnp.mean(jnp.array(final_test_accuracies))

print(f"\nFinal Test Loss: {final_test_loss:.4f}")
print(f"Final Test Accuracy: {final_test_accuracy:.4f}")

print("\n✅ Final model evaluation complete.")

# Section 7: Inference Demo
This final section demonstrates how to load the saved inference model and use it to classify new images. This is a crucial step for showcasing the practical application of the trained model.

In [ ]:
# @title 7.1 Loading the Saved Inference Model
# @markdown Load the model state saved previously for inference purposes.

# Define the path where the final model was saved.
final_model_dir = os.path.join(os.getcwd(), "final_model")

# This variable will hold our model instance ready for inference.
inference_model = None

# Check if the saved model directory exists.
if os.path.exists(final_model_dir) and os.listdir(final_model_dir):
    print(f"Restoring final model from '{final_model_dir}' for inference...")

    # --- Loading Model State with PyTreeCheckpointer ---
    # 1. Instantiate a PyTreeCheckpointer. This is simpler than CheckpointManager
    #    for single PyTree structures like a model's state.
    restorer = ocp.PyTreeCheckpointer()

    # 2. Create an 'abstract' model instance. This is necessary so Orbax knows
    #    the expected structure and types of the PyTree to restore.
    #    Crucially, the RNG streams ('params', 'dropout') must match the training setup.
    #    The actual keys don't matter for inference, but their names do for structure.
    abstract_rngs = nnx.Rngs(params=jax.random.PRNGKey(0), dropout=jax.random.PRNGKey(1))
    abstract_model = CNN(num_classes=NUM_CLASSES, rngs=abstract_rngs)

    # 3. Get the state structure from the abstract model.
    abstract_state = nnx.state(abstract_model)

    # 4. Restore the state. The `item` argument tells Orbax what structure to expect.
    restored_state = restorer.restore(final_model_dir, item=abstract_state)

    # 5. Update our `inference_model` instance with the restored state.
    #    `nnx.update` copies the restored values into the `inference_model`.
    inference_model = abstract_model # Use the abstract model structure
    nnx.update(inference_model, restored_state) # Update its state

    print("✅ Final model restored successfully for inference.")
else:
    print(f"❌ Final model directory not found at '{final_model_dir}'.")
    print("   Please ensure the training and final model saving cells have been run successfully.")

print("\nModel ready for inference if restoration was successful.")

In [ ]:
# @title 7.2 Download and Preprocess Test Images
# @markdown Define a helper function to fetch and prepare images for inference.

def download_and_preprocess_for_inference(url: str, target_size: tuple = (32, 32)):
    """
    Downloads an image from a URL, preprocesses it for the CNN model, and returns
    it as a JAX numpy array.

    Args:
        url: The URL of the image to download.
        target_size: The desired input size for the image (width, height).

    Returns:
        A JAX numpy array representing the preprocessed image, or None if an error occurs.
    """
    print(f"Attempting to download: {url}")
    try:
        # It's good practice to set a User-Agent for web requests.
        headers = {
            'User-Agent': 'JAX_Ecosystem_Demo/1.0 (Colab; example-contact@google.com)'
        }
        response = requests.get(url, headers=headers, timeout=10) # Add a timeout
        response.raise_for_status() # Raise an exception for bad status codes (e.g., 404)

        # Open image, convert to RGB (handling potential alpha channels), resize, and normalize.
        img = Image.open(BytesIO(response.content)).convert('RGB')
        # Using LANCZOS for high-quality downsampling.
        img = img.resize(target_size, Image.Resampling.LANCZOS)
        img_array = np.array(img).astype(np.float32) / 255.0

        # Convert the NumPy array to a JAX numpy array.
        return jnp.array(img_array)

    except requests.exceptions.RequestException as e:
        print(f"Error downloading image from {url}: {e}")
    except IOError as e:
        print(f"Error processing image from {url}: {e}. Ensure it's a valid image file.")
    except Exception as e:
        print(f"An unexpected error occurred for {url}: {e}")
    return None

print("Image download and preprocessing function defined.")

In [ ]:
# @title 7.3 Run Inference and Visualize Results
# @markdown Use the loaded model to predict classes for sample images and display the results.

# Define a dictionary of image URLs for demonstration.
# These are examples representing different superclasses.
image_urls_for_inference = {
    "Aquatic Mammals (Dolphin)": "https://upload.wikimedia.org/wikipedia/commons/9/9f/Bottlenose_Dolphin.jpg",
    "Fish (Goldfish)": "https://upload.wikimedia.org/wikipedia/commons/thumb/d/d5/Goldfish_in_bowl.jpg/1280px-Goldfish_in_bowl.jpg",
    "Flowers (Rose)": "https://upload.wikimedia.org/wikipedia/commons/2/28/Red_rose.jpg",
    "Fruit and Vegetables (Apple)": "https://upload.wikimedia.org/wikipedia/commons/thumb/1/15/Red_Apple.jpg/1024px-Red_Apple.jpg",
    "Household Furniture (Chair)": "https://upload.wikimedia.org/wikipedia/commons/thumb/0/06/A_Modern_Adirondack_Chair.jpg/1280px-A_Modern_Adirondack_Chair.jpg",
    "Large Carnivores (Wolf)": "https://upload.wikimedia.org/wikipedia/commons/thumb/5/5f/Kolm%C3%A5rden_Wolf.jpg/1280px-Kolm%C3%A5rden_Wolf.jpg",
    "Large Natural Outdoor Scenes (Forest)": "https://upload.wikimedia.org/wikipedia/commons/thumb/6/6b/Majestic_Mountain_View.jpg/1280px-Majestic_Mountain_View.jpg",
    "Vehicles 1 (Bicycle)": "https://upload.wikimedia.org/wikipedia/commons/thumb/e/e6/Bike_in_the_water_%2851312993027%29.jpg/1280px-Bike_in_the_water_%2851312993027%29.jpg",
    "Vehicles 2 (Train)": "https://upload.wikimedia.org/wikipedia/commons/thumb/1/1b/Amtrak_Acela_Express_2035_at_New_Haven_Union_Station.jpg/1280px-Amtrak_Acela_Express_2035_at_New_Haven_Union_Station.jpg" # Corrected URL for Train
}

# Proceed with inference only if the model was successfully loaded.
if inference_model is not None:
    print("\nRunning inference on sample images...")
    # Determine the layout for the plots. A 3x3 grid should be sufficient.
    num_images_to_show = min(len(image_urls_for_inference), 9) # Show up to 9 images
    fig, axes = plt.subplots(3, 3, figsize=(15, 15))
    axes = axes.flatten() # Flatten the 2D array of axes for easy iteration.
    current_ax_idx = 0

    # Iterate through the provided image URLs.
    for name, url in tqdm(image_urls_for_inference.items(), desc="Processing Inference Images"):
        # Download and preprocess the image.
        preprocessed_img_array = download_and_preprocess_for_inference(url)

        # If preprocessing was successful, proceed with inference.
        if preprocessed_img_array is not None:
            # Add a batch dimension: the model expects input of shape (batch_size, height, width, channels).
            input_tensor = jnp.expand_dims(preprocessed_img_array, axis=0)

            # --- Perform Inference ---
            # Run the model in evaluation mode (`training=False`) to ensure correct behavior
            # for layers like BatchNorm and Dropout.
            logits = inference_model(input_tensor, training=False)

            # Get the predicted class index (the one with the highest logit).
            predicted_class_index = jnp.argmax(logits, axis=-1).item()
            predicted_class_name = CIFAR100_SUPERCLASS_NAMES[predicted_class_index]

            # --- Display Results ---
            ax = axes[current_ax_idx]
            # Convert the JAX array to a NumPy array for plotting.
            ax.imshow(np.array(preprocessed_img_array)) # Display the image
            ax.set_title(f"{name}\nPred: {predicted_class_name}", fontsize=10)
            ax.axis('off') # Hide axes for a cleaner look
            current_ax_idx += 1

    # Hide any unused subplots if fewer than 9 images were processed.
    for i in range(current_ax_idx, len(axes)):
        axes[i].axis('off')

    plt.tight_layout(rect=[0, 0.03, 1, 0.95]) # Adjust layout to make space for suptitle
    plt.suptitle("Image Classification Inference Results", fontsize=16)
    plt.show()

    print("\nInference demonstration complete.")
else:
    print("\nSkipping inference demo as the model could not be loaded.")

print("\nJAX Ecosystem Demonstration Complete!")

In [ ]:
# @title 7.4 Testing with Custom Images
# @markdown Instructions for testing the model with your own images.

# ## Testing with Custom Images

# To test the model with your own images, you can modify the `image_urls_for_inference` dictionary.
# Replace the existing URLs with the URLs of the images you want to classify.
#
# For example, to test with a specific image of a dog:
#
# ```python
# image_urls_for_inference = {
#     "Custom Dog Image": "YOUR_DOG_IMAGE_URL_HERE"
# }
# ```
#
# **Important Considerations:**
# *   **Image Format:** Ensure your images are in a common format (like JPEG, PNG) and are accessible via a direct URL.
# *   **Image Size:** The model expects images of size 32x32 pixels. The `download_and_preprocess_for_inference` function handles resizing, but very large or small images might be distorted.
# *   **Content Alignment:** The model was trained on CIFAR-100 superclasses. Images that are visually similar to these classes will likely be classified correctly. Images vastly different from the training data may yield unpredictable results.
#
# After modifying the `image_urls_for_inference` dictionary, simply re-run the "Run Inference and Visualize Results" cell (Section 7.3) to see the predictions for your custom images.

print("Instructions for testing with custom images provided.")

# Conclusion and Further Exploration
### Summary of Demonstrated Capabilities
This notebook has walked you through a complete image classification pipeline using the JAX ecosystem, highlighting:
* JAX Performance: Demonstrating the power of JAX for accelerated numerical computation through its automatic differentiation and JIT compilation.
* Flax NNX: Showcasing its declarative and stateful approach to building complex neural network architectures efficiently.
* Optax: Illustrating its flexibility in defining sophisticated optimization strategies, including learning rate scheduling.
* Orbax Checkpointing: Highlighting its crucial role in managing model state for reproducibility and fault tolerance, essential for research.
* End-to-End Workflow: Providing a practical example from data loading and augmentation to model training, evaluation, and inference.
### Next Steps for Developers and Researchers
This demonstration serves as a foundation. Here are some ideas for extending this work:
* Hyperparameter Tuning: Experiment with different learning rates, batch sizes, optimizers, and network architectures.
* Advanced Augmentations: Explore more sophisticated data augmentation techniques (e.g., Mixup, CutMix) for potentially better performance.
* Different Architectures: Implement and compare other JAX-compatible architectures like ResNets, Vision Transformers, etc.
* Distributed Training: Investigate how JAX's pmap and shard_map can be used for efficient training across multiple devices (GPUs/TPUs).
* Custom Datasets: Adapt the data loading and preprocessing pipeline to work with your own datasets.
* Reproducibility: Use Orbax to rigorously save and load experiments, ensuring your research is verifiable.
We hope this notebook provides a valuable starting point for your JAX-based machine learning projects!